In [1]:
from dotenv import load_dotenv
from langchain_community.document_loaders import TextLoader
import os

load_dotenv('./.env')
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

loaders=[TextLoader("./Data/How_to_invest_money.txt", encoding='utf8')]

docs = []
for loader in loaders:
    docs.extend(loader.load())

C:\Users\Dohy\AppData\Local\Temp\ipykernel_9400\362416108.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader
c:\Users\Dohy\Desktop\dohy\RAG_master\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_classic.storage import InMemoryStore
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 부모 문서 생성을 위한 텍스트 분할기
parent_splitter = RecursiveCharacterTextSplitter(chunk_size=1000)
# 자식 문서 생성을 위한 텍스트 분할기(부모보다 작은 크기로 설정)
child_splitter = RecursiveCharacterTextSplitter(chunk_size=200)

# 자식 문서 인덱싱을 위한 벡터 저장소
vectorstore = Chroma(
    collection_name="split_parents", embedding_function=OpenAIEmbeddings()
)

# 부모 문서 저장을 위한 저장소
store = InMemoryStore() # RAM에 데이터를 저장하는 방식, 빠른 읽기와 쓰기가 가능하지만 프로그램 종료 시 데이터가 손실

# ParentDocumentRetriever 설정
retriever = ParentDocumentRetriever(
    vectorstore = vectorstore, # 자식 문서 저장소
    docstore = store, # 부모 문서 저장소
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
)

# 문서 추가
retriever.add_documents(docs)

# 부모 문서 수 확인
print(f"Number of parnet docs: {len(list(store.yield_keys()))}") # yield_keys를 사용하여 저장된 모든 부모 문서의 키를 가져옴, {"ID" : 부모문서} 이런 식으로 저장

Number of parnet docs: 219


In [ ]:
# 질문 정의
query = "What are the types of investments?"

# 연관 문서 수집
retrieved_docs = retriever.invoke(query) # 1. vectorstore.similarity_search로 자식 청크 검색  ← 여기가 벡터 검색
                                         # 2. 그 자식 문서의 부모 ID를 찾음
                                         # 3. InMemoryStore에서 부모 문서를 꺼냄  ← 여기는 ID로 직접 꺼내기 
                                         # 4. 부모 문서 반환

# 첫 번쨰 연관 문서 출력
print(f"Parent Document: {retrieved_docs[0].page_content}")

Parent Document: There are five chief points to be considered in the selection of all
forms of investment. These are: (1) safety of principal and interest;
(2) rate of income; (3) convertibility into cash; (4) prospect of
appreciation in intrinsic value; (5) stability of market price.

Keeping these five general factors in mind, the present chapter will
discuss real-estate mortgages as a form of investment, both as adapted
to the requirements of private funds and of a business surplus.


In [4]:
# 자식 문서 검색
query = "What are the types of investments?"
sub_docs = vectorstore.similarity_search(query)
print(f"Child Doc: {sub_docs[0].page_content}")

Child Doc: forms of investment. These are: (1) safety of principal and interest;
(2) rate of income; (3) convertibility into cash; (4) prospect of
appreciation in intrinsic value; (5) stability of market price.


In [ ]:
# 다중 질의 생성
import logging
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_classic.retrievers import MultiQueryRetriever
from langchain_openai import ChatOpenAI
logging.basicConfig()
logging.getLogger("langchain_classic.retrievers.multi_query").setLevel(logging.INFO)

loaders = [TextLoader("./Data/How_to_invest_money.txt", encoding='utf8')]

docs = []
for loader in loaders:
    docs.extend(loader.load())

1


In [ ]:
# 문서 생성을 위한 텍스트 분할기 정의
recursive_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)

# 문서 분할
split_docs = recursive_splitter.split_documents(docs)

embeddings = OpenAIEmbeddings()
vectorstore = Chroma.from_documents(documents=split_docs, embedding=embeddings)

llm = ChatOpenAI(model='gpt-4o-mini', temperature=0.2)

# MultiQueryRetriever 실행
retriever = MultiQueryRetriever.from_llm(  # 커스텀 프롬프트를 따로 주지 않으면, 3개의 다중 쿼리 생성
    retriever=vectorstore.as_retriever(), # 기본 검색기(벡터DB)
    llm=llm, # 다중 쿼리 생성을 위한 LLM
)

query = "주식 투자를 처음 시작하려면 어떻게 해야 하나요?"

# 결과 검색
unique_docs = retriever.invoke(query)
print(f"\n결과:{len(unique_docs)}개의 문서가 검색됐습니다.")

INFO:langchain_classic.retrievers.multi_query:Generated queries: ['주식 투자를 시작하는 데 필요한 첫 단계는 무엇인가요?  ', '주식 투자 초보자가 알아야 할 기본 사항은 무엇인가요?  ', '주식 시장에 처음 진입할 때 유의해야 할 점은 무엇인가요?']



결과:2개의 문서가 검색됐습니다.


AttributeError: 'RunnableSequence' object has no attribute 'prompt'

In [ ]:
# MultiQueryRetriever 기본 프롬프트 확인
from langchain_classic.retrievers.multi_query import DEFAULT_QUERY_PROMPT
print(DEFAULT_QUERY_PROMPT.template)

You are an AI language model assistant. Your task is
    to generate 3 different versions of the given user
    question to retrieve relevant documents from a vector  database.
    By generating multiple perspectives on the user question,
    your goal is to help the user overcome some of the limitations
    of distance-based similarity search. Provide these alternative
    questions separated by newlines. Original question: {question}


In [7]:
from langchain_classic.chains import RetrievalQA

# RetrievalQA 체인 설정
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type='stuff',
    retriever=retriever,
    return_source_documents=True
)

# 질문에 대한 답변 생성
result = qa_chain.invoke({"query":query})

# 결과 출력
print("답변: ",result['result'])
print('\n사용된 문서\n')
for doc in result['source_documents']:
    print(doc.page_content)

INFO:langchain_classic.retrievers.multi_query:Generated queries: ['주식 투자에 처음 입문할 때 어떤 단계부터 시작해야 할까요?  ', '주식 투자를 시작하기 위해 필요한 기본 지식이나 팁은 무엇인가요?  ', '초보자가 주식 투자에 성공하기 위해 알아야 할 사항은 어떤 것들이 있나요?']


답변:  주식 투자를 처음 시작하려면 다음과 같은 단계를 고려해 보세요:

1. **기본 지식 습득**: 주식 시장의 기본 개념, 주식과 채권의 차이, 투자 원칙 등을 학습하세요.

2. **투자 목표 설정**: 자신의 투자 목표와 기간을 명확히 하세요. 예를 들어, 단기 수익을 원하시는지, 장기적인 자산 증식을 원하시는지 결정합니다.

3. **재무 상태 점검**: 투자할 자금을 마련하기 전에 자신의 재무 상태를 점검하고, 비상금이나 필수 지출을 고려하세요.

4. **증권 계좌 개설**: 주식 거래를 위해 증권사에 계좌를 개설하세요. 여러 증권사의 수수료와 서비스 등을 비교해 보세요.

5. **시장 조사**: 투자할 기업이나 산업에 대한 조사를 통해 정보를 수집하고, 주식의 가치를 평가하세요.

6. **분산 투자**: 리스크를 줄이기 위해 다양한 주식에 분산 투자하는 것이 좋습니다.

7. **장기적인 관점 유지**: 주식 시장은 변동성이 크기 때문에 단기적인 가격 변동에 일희일비하지 않고 장기적인 관점을 유지하는 것이 중요합니다.

이러한 단계를 통해 주식 투자를 시작할 수 있습니다.

사용된 문서

G. G. H.




HOW TO INVEST MONEY




I

GENERAL PRINCIPLES OF INVESTMENT
which point to the beginning of a pronounced upward swing in securities,
and if he can equally recognize the signs which indicate that the
movement has culminated, he can liquidate the securities which he bought
at the inception of the rise or transfer them to some short-term issues
whose near approach to maturity will render them stable in price,
allo

In [8]:
for i, doc in enumerate(result['source_documents'],1):
    print(f"--- 문서 {i} (길이: {len(doc.page_content)}) ---")
    print(repr(doc.page_content))   # repr로 보면 줄바꿈/공백까지 다 보임
    print(doc.metadata)
    print()

--- 문서 1 (길이: 72) ---
'G. G. H.\n\n\n\n\nHOW TO INVEST MONEY\n\n\n\n\nI\n\nGENERAL PRINCIPLES OF INVESTMENT'
{'source': './Data/How_to_invest_money.txt'}

--- 문서 2 (길이: 641) ---
'which point to the beginning of a pronounced upward swing in securities,\nand if he can equally recognize the signs which indicate that the\nmovement has culminated, he can liquidate the securities which he bought\nat the inception of the rise or transfer them to some short-term issues\nwhose near approach to maturity will render them stable in price,\nallowing the downward swing to proceed without disturbing him. It is not\nexpected, of course, that the average business man will be able to\nrealize completely this ideal of investment, but it is hoped that the\nfollowing analysis will give him a clearer conception of the principles\ninvolved.'
{'source': './Data/How_to_invest_money.txt'}

--- 문서 3 (길이: 974) ---
'After learning how to judge the value of every form of investment, a man\nmay still be unsuccessful

In [ ]:
# 가상 문서 임베딩(HYDE)
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loaders = [TextLoader("./Data/How_to_invest_money.txt", encoding='utf8')]
docs = []
for loader in loaders:
    docs.extend(loader.load())

recursive_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
split_docs = recursive_splitter.split_documents(docs)
embeddings=OpenAIEmbeddings()
vectorstore = Chroma.from_documents(documents=split_docs, embedding=embeddings)
retriever = vectorstore.as_retriever()

[Document(metadata={'source': './Data/How_to_invest_money.txt'}, page_content='\ufeffThe Project Gutenberg eBook of How to Invest Money\n    \nThis ebook is for the use of anyone anywhere in the United States and\nmost other parts of the world at no cost and with almost no restrictions\nwhatsoever. You may copy it, give it away or re-use it under the terms\nof the Project Gutenberg License included with this ebook or online\nat www.gutenberg.org. If you are not located in the United States,\nyou will have to check the laws of the country where you are located\nbefore using this eBook.\n\nTitle: How to Invest Money\n\nAuthor: George Garr Henry\n\nRelease date: November 28, 2010 [eBook #34463]\n                Most recently updated: January 7, 2021\n\nLanguage: English\n\nCredits: Produced by Julia Neufeld and the Online Distributed\n        Proofreading Team at https://www.pgdp.net (This file was\n        produced from images generously made available by The\n        Internet Archive/Am

In [29]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import OpenAIEmbeddings
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 가상 문서 생성 체인
def create_virtual_doc_chain():
    system="당신은 고도로 숙련된 AI입니다."
    user="""
    주어진 질문 '{query}'에 대해 직접적으로 답변하는 가상의 문서를 생성하세요.
    문서의 크기는 {chunk_size} 글자 언저리여야 합니다.
    """
    prompt=ChatPromptTemplate.from_messages([
        ("system",system),
        ("human",user)
    ])

    llm = ChatOpenAI(model='gpt-4o', temperature=0.2)
    return prompt | llm | StrOutputParser()


# 문서 검색 체인, 가상의 답변을 기반으로 벡터 DB에서 가장 유사한 문서를 찾아서 반환
def create_retrieval_chain():
    #return RunnableLambda(lambda x: retriever.get_relevant_documents(x['virtual_doc']))
    return RunnableLambda(lambda x: retriever.invoke(x['virtual_doc']))
# 유틸리티 함수, 반환한 문서에서 메타데이터는 제외하고 순수한 문서 내용만 추출
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# 최종 응답 생성 체인
def create_final_response_chain():
    final_prompt = ChatPromptTemplate.from_template("""
                    다음 정보와 질문을 바탕으로 답변해주세요:
                    컨텍스트: {context}
                                                    
                    질문: {question}
                                                    
                    답변:
                    """)
    final_llm = ChatOpenAI(model='gpt-4o', temperature=0.2)
    return final_prompt | final_llm

In [30]:
# 로깅을 위한 유틸리티 함수, 각 단계에서 입력과 출력을 프린트
def print_input_output(input_data, output_data, step_name):
    print(f"\n--- {step_name} ---")
    print(f"Input: {input_data}")
    print(f"Output: {output_data}")
    print("-"*50)

def create_pipeline_with_logging():
    virtual_doc_chain = create_virtual_doc_chain()
    retrieval_chain = create_retrieval_chain()
    final_response_chain = create_final_response_chain()

    # 가상 문서 생성 단계
    def virtual_doc_step(x): # x는 사용자의 질문을 포함하는 딕셔너리
        result = {'virtual_doc': virtual_doc_chain.invoke({
            "query":x['question'],
            "chunk_size" : 200
        })}
        print_input_output(x, result, "Virtual Doc Generation")
        return {**x, **result} # 언패킹하여 하나의 딕셔너리로 생성
    
    # 문서 검색 단계
    def retrieval_step(x): # x는 이전 단계에서 생성된 가상 문서와 원본 질문이 포함된 딕셔너리
        result = {"retrieved_docs": retrieval_chain.invoke(x)}
        print_input_output(x, result, "Document Retrieval")
        return {**x, **result} # 언패킹하여 하나의 딕셔너리로 생성
    
    # 컨텍스트 포매팅 단계
    def context_formatting_step(x): # x는 이전 단계에서 생성된 검색된 문서들이 포함된 딕셔너리
        result = {"context": format_docs(x['retrieved_docs'])}
        print_input_output(x, result, "Context Formatting")
        return {**x, **result} # 언패킹하여 하나의 딕셔너리로 생성
    
    # 최종 응답 생성 단계
    def final_response_step(x): # x는 컨텍스트와 원본 질문을 포함한 딕셔너리
        result = final_response_chain.invoke(x)
        print_input_output(x, result, "Final Response Generation")
        return result
    
    # 전체 파이프라인 구성
    pipeline= (
        RunnableLambda(virtual_doc_step)
        | RunnableLambda(retrieval_step)
        | RunnableLambda(context_formatting_step)
        | RunnableLambda(final_response_step)
    )
    
    return pipeline

# 파이프라인 객체 생성
pipeline = create_pipeline_with_logging()
query = "주식 시장의 변동성이 높을 때 투자 전략은 무엇인가요?"
response = pipeline.invoke({"question":query})
print(f"최종 답변:{response.content}")


--- Virtual Doc Generation ---
Input: {'question': '주식 시장의 변동성이 높을 때 투자 전략은 무엇인가요?'}
Output: {'virtual_doc': '주식 시장의 변동성이 높을 때는 신중한 투자 전략이 필요합니다. 첫째, 포트폴리오를 다각화하여 리스크를 분산시키는 것이 중요합니다. 둘째, 방어적인 주식이나 배당주에 투자하여 안정성을 확보할 수 있습니다. 셋째, 현금을 보유하여 기회를 기다리는 것도 전략 중 하나입니다. 마지막으로, 장기적인 관점에서 시장을 바라보며 단기 변동에 휘둘리지 않는 것이 중요합니다. 이러한 전략을 통해 변동성 높은 시장에서도 안정적인 수익을 추구할 수 있습니다.'}
--------------------------------------------------

--- Document Retrieval ---
Input: {'question': '주식 시장의 변동성이 높을 때 투자 전략은 무엇인가요?', 'virtual_doc': '주식 시장의 변동성이 높을 때는 신중한 투자 전략이 필요합니다. 첫째, 포트폴리오를 다각화하여 리스크를 분산시키는 것이 중요합니다. 둘째, 방어적인 주식이나 배당주에 투자하여 안정성을 확보할 수 있습니다. 셋째, 현금을 보유하여 기회를 기다리는 것도 전략 중 하나입니다. 마지막으로, 장기적인 관점에서 시장을 바라보며 단기 변동에 휘둘리지 않는 것이 중요합니다. 이러한 전략을 통해 변동성 높은 시장에서도 안정적인 수익을 추구할 수 있습니다.'}
Output: {'retrieved_docs': [Document(metadata={'source': './Data/How_to_invest_money.txt'}, page_content='For the successful investment of money, however, a good deal more is\nrequired than the mere ability to select a safe security. That i

In [ ]:
# 희소 검색 By BM25
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.retrievers import BM25Retriever
from kiwipiepy import Kiwi
from langchain_classic.chains import RetrievalQA, ConversationalRetrievalChain
from langchain_openai import ChatOpenAI

file_path = './Data/투자설명서.pdf'
loader = PyPDFLoader(file_path)
doc_splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=200)
docs = loader.load_and_split(doc_splitter)

kiwi_tokenizer = Kiwi()

def kiwi_tokenize(text):
    return [token.form for token in kiwi_tokenizer.tokenize(text)]

bm25_retriever = BM25Retriever.from_documents(docs, preprocess_func=kiwi_tokenize)
bm25_retriever.k = 2

# 관련있는 문서 수집 후, ChatGPT로 최종 답변까지 수행하는 체인 생성
qa_chain = RetrievalQA.from_chain_type(
    llm=ChatOpenAI(model='gpt-4o', temperature=0.2),
    chain_type='stuff',
    retriever=bm25_retriever,
    return_source_documents=True
)

qa_chain.invoke("이 회사가 발행한 주식의 총 발행량이 어느 정도야?")

{'query': '이 회사가 발행한 주식의 총 발행량이 어느 정도야?',
 'result': '이 회사의 총 발행주식수는 13,602,977주입니다.',
 'source_documents': [Document(metadata={'producer': 'iText® 5.5.9 ©2000-2015 iText Group NV (AGPL-version)', 'creator': 'PyPDF', 'creationdate': '2024-06-26T16:15:14+09:00', 'moddate': '2024-06-26T16:15:14+09:00', 'source': './Data/투자설명서.pdf', 'total_pages': 514, 'page': 341, 'page_label': '342'}, page_content='당사는 투기적 목적으로 파생금융상품을 포함한 금융상품계약을 체결하거나 거래하지 않습\n니다. \n  \n바. 파생거래 \n  \n당사가 2021년 3월 19일에 발행한 제2회 무기명식 무보증 사모 전환사채에는 조기상환청구\n권(Put option)과 중도상환청구권(Call option)이 포함되어 있습니다. 당분기말 현재 전환사\n채의 내용은 다음과 같습니다. \n구 \xa0분 내역\n사채의 종류 제2회 무기명식 무보증 사모 전환사채\n발행가액 (단위: 천원) 19,000,000\n미상환사채 권면총액 (단위: 천원\n)(*1) 7,600,000\n이자지급조건 없음\n연이자율 0.0%\n보장수익률 0.0%\n발행일 2021년 3월 19일\n상환일 2026년 3월 19일\n전환청구기간\n시작일 2022년 3월 19일\n종료일 2026년 2월 19일\n전환에 따라\n발행할 주식\n종류\n주식회사 셀리드 기명식 보통주\n식\n전환가액(원/주)(*2) 4,515\n주식수(*2) 1,683,277주\n전환비율(%) 100\n전환가액의 조정조건\n1.본건 전환사채의 전환가액을 하회하는 발행가액, 전환가액 또\n는 행사가액으로 유상증자 또는 그에 준하는 주식연계증권(전환

In [9]:
# 밀집 검색 By FAISS
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_classic.chains import RetrievalQA, ConversationalRetrievalChain

file_path="./Data/투자설명서.pdf"
loader =PyPDFLoader(file_path)
doc_splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=200)
docs=loader.load_and_split(doc_splitter)

embedding = OpenAIEmbeddings(model='text-embedding-3-large')

faiss_store = FAISS.from_documents(docs, embedding)
faiss_store.save_local("./faissDB/")

persist_directory = "./faissDB/"
vectordb = FAISS.load_local(persist_directory, embeddings=embedding, allow_dangerous_deserialization=True)

faiss_retriever = vectordb.as_retriever(search_kwargs={"k":4})

qa_chain = RetrievalQA.from_chain_type(
    llm=ChatOpenAI(temperature=0.2, model='gpt-4o'),
    chain_type = "stuff",
    retriever = faiss_retriever,
    return_source_documents=True
)

qa_chain.invoke("이 회사가 발행한 주식의 총 발행량이 어느 정도야?")

{'query': '이 회사가 발행한 주식의 총 발행량이 어느 정도야?',
 'result': '이 회사가 발행한 주식의 총 발행량은 13,602,977주입니다.',
 'source_documents': [Document(id='f80ec3ed-66cb-47ad-a030-275b152b21a8', metadata={'producer': 'iText® 5.5.9 ©2000-2015 iText Group NV (AGPL-version)', 'creator': 'PyPDF', 'creationdate': '2024-06-26T16:15:14+09:00', 'moddate': '2024-06-26T16:15:14+09:00', 'source': './Data/투자설명서.pdf', 'total_pages': 514, 'page': 327, 'page_label': '328'}, page_content="(*1) 정부R&D예산 축소로 인한 사업비 규모가 변경되었습니다. 기타 세부 사항은 2024년\n3월 15일 기 제출한 기타경영사항을 참조하시길 바랍니다.  \n \n(*2) 당사는 2024년 3월 12일 이사회 결의에 의거하여 주식회사 포베이커를 소규모 합병의\n형태로 흡수합병을 결정하였습니다. \xa02024년 5월 14일을 합병기일로 하여 흡수합병을 완료\n하였습니다. 자세한 내용은 2024년 3월 12일 '주요사항보고서(회사합병결정)' 과 \xa02024년\n5월 21일 '합병등종료보고서(합병)' 공시를 참고하시기 바랍니다. \n  \n3. 자본금 변동사항 \n \n당사는 본 보고서 작성기준일 현재 기업공시서식 작성기준에 따른 소규모 기업에 해당하므\n로 상기 자본금 변동추이를 기재하지 않았습니다. \n  \n4. 주식의 총수 등 \n  \n가. 주식의 총수 \n  \n주식의 총수 현황 \n설\n투자 차입금에 대한 담보 제공\n2. 채권자 : 하나캐피탈(주) 외\n3. 채무금액 : 32,560백만원\n4. 담보설정금액 : 42,328백만원\n5. 담보제공재산 : 

In [11]:
retrieved = faiss_retriever.invoke("이 회사가 발행한 주식의 총 발행량이 어느 정도야?")
for i, doc in enumerate(retrieved):
    print(f"--- 청크 {i} ---")
    print(doc.page_content[:300])
    print()

--- 청크 0 ---
(*1) 정부R&D예산 축소로 인한 사업비 규모가 변경되었습니다. 기타 세부 사항은 2024년
3월 15일 기 제출한 기타경영사항을 참조하시길 바랍니다.  
 
(*2) 당사는 2024년 3월 12일 이사회 결의에 의거하여 주식회사 포베이커를 소규모 합병의
형태로 흡수합병을 결정하였습니다.  2024년 5월 14일을 합병기일로 하여 흡수합병을 완료
하였습니다. 자세한 내용은 2024년 3월 12일 '주요사항보고서(회사합병결정)' 과  2024년
5월 21일 '합병등종료보고서(합병)' 공시를 참고하시기 바랍니다. 
  
3. 자본금 

--- 청크 1 ---
본 증권신고서 "제1부.III.투자위험요소-2.회사위험-아." 부분에 기재된 바와 같이 당사는 현
재 임직원 등에 부여한 주식매수선택권 210,262주가 잔존하고 있습니다. 향후 당사 주식수
에 변동이 생길 수 있는 기발행된 잠재주식은 주식매수선택권에 해당됩니다.(주1)  
  
 
에 출회될 경우 물량 출회에 따른 주가 하락 위험이 존재합니다. 투자자 여러분들께서는 이 점을 반
드시 유의하여 주시기 바랍니다.
(주1) 전환사채 잔액 76억원(권면금액)은 2024년 6월 19일 조기상환 완료하였습니다.
[주식매수선택권 내역]
(현재

--- 청크 2 ---
※ 참고 : 구주주 1주당 배정비율 산출 근거 
  
3. 공모가격 결정방법 
  
「증권의 발행 및 공시 등에 관한 규정」 제5-18조에 의거 주주배정 증자시 가격산정 절차
폐지 및 가격산정의 자율화에 따라 발행가격을 자유롭게 산정할 수 있으나, 시장 혼란 우려
및 기존 관행 등으로 (구)「유가증권의 발행 및 공시 등에 관한 규정」 제57조를 일부 준용하
여 아래와 같이 발행가액을 산정합니다. 
  
가. 1차 발행가액: 신주배정기준일 전 제3거래일을 기산일로 하여 코스닥시장에서 성립된 거
래대금을 거래량으로 가중산술평균한 1개월

--- 청크 3 ---
다. 경영권 경쟁 
 
 당사는 본 보고서 작성기준일 현재까지 경영권과 관련

In [15]:
# 앙상블 검색
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.retrievers import BM25Retriever
from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_classic.retrievers import EnsembleRetriever
from langchain_classic.chains import RetrievalQA, ConversationalRetrievalChain
from langchain_openai import ChatOpenAI
from kiwipiepy import Kiwi

file_path = './Data/투자설명서.pdf'
loader = PyPDFLoader(file_path)
doc_splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=200)
docs = loader.load_and_split(doc_splitter)

kiwi_tokenizer = Kiwi()
def kiwi_tokenize(text):
    return [token.form for token in kiwi_tokenizer.tokenize(text)]

bm25_retriever = BM25Retriever.from_documents(docs, preprocess_func=kiwi_tokenize)
bm25_retriever.k = 4

embedding = OpenAIEmbeddings(model='text-embedding-3-large')

faiss_store = FAISS.from_documents(docs, embedding)
faiss_store.save_local("./faissDB/")

persist_directory = "./faissDB/"
vectordb = FAISS.load_local(persist_directory, embeddings=embedding, allow_dangerous_deserialization=True)

faiss_retriever = vectordb.as_retriever(search_kwargs={"k":4})
ensemble_retriever = EnsembleRetriever(retrievers=[bm25_retriever, faiss_retriever], weights=[0.5,0.5])

qa_chain = RetrievalQA.from_chain_type(
    llm=ChatOpenAI(temperature=0.2, model='gpt-4o'),
    chain_type="stuff",
    retriever = ensemble_retriever,
    return_source_documents=True
)

qa_chain.invoke("이 회사가 발행한 주식의 총 발행량은 어느 정도야?")

{'query': '이 회사가 발행한 주식의 총 발행량은 어느 정도야?',
 'result': '이 회사가 발행한 주식의 총 발행량은 13,602,977주입니다.',
 'source_documents': [Document(metadata={'producer': 'iText® 5.5.9 ©2000-2015 iText Group NV (AGPL-version)', 'creator': 'PyPDF', 'creationdate': '2024-06-26T16:15:14+09:00', 'moddate': '2024-06-26T16:15:14+09:00', 'source': './Data/투자설명서.pdf', 'total_pages': 514, 'page': 341, 'page_label': '342'}, page_content='당사는 투기적 목적으로 파생금융상품을 포함한 금융상품계약을 체결하거나 거래하지 않습\n니다. \n  \n바. 파생거래 \n  \n당사가 2021년 3월 19일에 발행한 제2회 무기명식 무보증 사모 전환사채에는 조기상환청구\n권(Put option)과 중도상환청구권(Call option)이 포함되어 있습니다. 당분기말 현재 전환사\n채의 내용은 다음과 같습니다. \n구 \xa0분 내역\n사채의 종류 제2회 무기명식 무보증 사모 전환사채\n발행가액 (단위: 천원) 19,000,000\n미상환사채 권면총액 (단위: 천원\n)(*1) 7,600,000\n이자지급조건 없음\n연이자율 0.0%\n보장수익률 0.0%\n발행일 2021년 3월 19일\n상환일 2026년 3월 19일\n전환청구기간\n시작일 2022년 3월 19일\n종료일 2026년 2월 19일\n전환에 따라\n발행할 주식\n종류\n주식회사 셀리드 기명식 보통주\n식\n전환가액(원/주)(*2) 4,515\n주식수(*2) 1,683,277주\n전환비율(%) 100\n전환가액의 조정조건\n1.본건 전환사채의 전환가액을 하회하는 발행가액, 전환가액 또\n는 행사가액으로 유상증자 또는 그에 준하는 주식연

In [2]:
# 고성능 LLM 기반 리랭킹
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from pydantic import BaseModel, Field
from langchain_core.prompts import PromptTemplate
from langchain_community.docstore.document import Document
from typing import List, Dict, Any, Tuple
from textwrap import dedent
from langchain_core.output_parsers import JsonOutputParser

file_path="./Data/투자설명서.pdf"
loader = PyPDFLoader(file_path)

doc_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=100)
docs = loader.load_and_split(doc_splitter)

embedding = OpenAIEmbeddings(model='text-embedding-3-large')

faiss_store = FAISS.from_documents(docs,embedding)
persist_directory="./faissDB/"
faiss_store.save_local(persist_directory)
vectordb = faiss_store.load_local(persist_directory, embeddings=embedding, allow_dangerous_deserialization=True)

# 관련성 점수를 나타내는 데이터 모델 정의
class RelevanceScore(BaseModel):
    relevance_score: float = Field(description="문서가 쿼리와 얼마나 관련이 있는지 나타내는 점수") # :float, description은 모델에게 가는 프롬프트

# 사용자의 질문과 초기 검색 문서들을 입력받고, 리랭킹이 수행된 문서 리스트를 반환하는 역할
def reranking_documents(query: str, docs: List[Document], top_n:int=2) -> List[Document]:
    parser = JsonOutputParser(pydantic_object=RelevanceScore) # LLM의 출력을 RelevanceScore 객체로 파싱
    human_message_prompt = PromptTemplate(
    template ="""
    1점부터 10점까지 점수를 매겨, 다음 문서가 질문과 얼마나 관련이 있는지 평가해주세요. 단순히 키워드가 일치하는 것이 아니라 쿼리의 구체적인 맥락과 의도를 고려하세요.
    {format_instructions}
    question:{query}
    document:{doc}
    relevance_score:""",
    input_variables=['query','doc'], # invoke 때 채워짐
    partial_variables={'format_instructions':parser.get_format_instructions()} # 프롬프트를 만들 때 미리 채움
    )

    llm = ChatOpenAI(temperature=0, model_name="gpt-4o", max_tokens=3000)
    chain = human_message_prompt | llm | parser
    scored_docs=[]
    for doc in docs:
        input_data = {'query':query, 'doc':doc.page_content} # 여기서 매개변수 query를 프롬프트의 값으로 넣어줌
        try:
            score = chain.invoke(input_data)['relevance_score']
            score = float(score) # RelevanceScore에서 float로 결과를 반환하라고 모델한테 알렸지만, 모델이 float로 안주는 확률이 있으니 float로 강제 형변환
        except Exception as e: # 에러난 것을 e 객체에 담음
            print(f"오류 발생: {str(e)}")
            default_score = 5.0 # 기본 점수
            print(f"기본 점수 {default_score}점을 사용합니다.")
            score = default_score
        scored_docs.append((doc,score))

    reranked_docs = sorted(scored_docs, key=lambda x: x[1], reverse=True)
    return [doc for doc, _ in reranked_docs[:top_n]]


query = "이 회사의 2022년 영업손실이 정확히 얼마야?"
initial_docs = vectordb.similarity_search(query, k=4)
reranked_docs = reranking_documents(query, initial_docs)

# 4개 초기 검색 결과 출력
print(f"Query: {query}\n\n")
print("Top initial documents:")
for i, doc in enumerate(initial_docs):
    print(f"\nDocument {i+1}:")
    print(doc.page_content)

# 리랭킹 결과 출력
print("\n\nTop reranked documents:")
for i,doc in enumerate(reranked_docs):
    print(f"\nDocument {i+1}:")
    print(doc.page_content)

C:\Users\Dohy\AppData\Local\Temp\ipykernel_28376\2034095117.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Query: 이 회사의 2022년 영업손실이 정확히 얼마야?


Top initial documents:

Document 1:
이 주요 원인으로 작용하여 당기순손실 228.7억원이 발생하였습니다. 
 
이어, 당사는 2023년에는 매출이 발생하지 않았으며, 종업원급여 40.9억원, 유무형자산상각
비 26.3억원이 발생하였으며, 연구개발 및 임상시험 지속 과정에서 지급수수료가 30.5억원
발생하는 등 총 영업비용 122억원이 발생하였습니다. 이에 2023년 영업손실 122억원이 발
생하였으며, 금융손익 및 기타손익 반영 후 당기순손실 116.1억원이 발생하였습니다. 한편,

Document 2:
이어, 당사는 2022년 GMP시설의 위탁생산계약 매출 약 4.8억원이 발생하였으나, 종업원급
여 43.9억원, 유무형자산상각비 25.3억원이 발생하였으며, 연구개발 및 임상시험 지속 과정
에서 지급수수료가 54.8억원 발생하는 등 총 영업비용 154억원이 발생하였습니다. 이에
2022년 영업손실 149억원이 발생하였으며, 한편, 2022년 5월 금융감독원에서 공표한 '전환
사채 콜옵션 회계처리' 감독지침에 따라 당사는 2021년 3월 19일 발행한 제2회 전환사채의

Document 3:
하여 2021년 영업손실 130.1억원, 2022년 영업손실 149.1억원, 2023년 영업손실 122억원, 2024년
1분기 영업손실 24.2억원이 발생하였습니다. 또한 영업 외적 측면에서도, 금융비용 등의 발생 영향
으로 인해 2021년 당기순손실 130.7억원, 2022년 당기순손실 228.7억원, 2023년 당기순손실
116.1억원, 2024년 1분기 당기순손실 32.9억원이 발생하는 등 지속적인 적자 구조를 면하지 못하고
있습니다.따라서 당사의 파이프라인에서 임상 성공을 통한 기술이전, 상품화 성공 등의 성과를 이루

Document 4:
2020년 11월 GMP 완공에 따라 2021년에는 소모품비 21.8억원, 유무형자산상각비 22.4억
원이 발생하였으며, 코로나19 예방

In [3]:
# 사용자 질문에 대한 검색과 리랭킹을 수행
from langchain_core.retrievers import BaseRetriever
from langchain_classic.chains import RetrievalQA
from pydantic import ConfigDict
# CustomRetriever 체인 생성
class CustomRetriever(BaseRetriever, BaseModel):
    model_config = ConfigDict(arbitrary_types_allowed=True)# Pydantic이 모르는 타입도 그냥 pass
    vectorstore: Any = Field(description="Retrieval을 위한 벡터 저장소")
    '''
    구 버전
    class Config:
        arbitrary_types_allowed = True # Pydantic이 모르는 타입도 그냥 pass
    '''
    
    # num_docs 파라미터로 리랭킹 후 반환할 최종 문서의 수 정의
    def _get_relevant_documents(self, query:str, num_docs=2) -> List[Document]:
        initial_docs = self.vectorstore.similarity_search(query, k=4)
        return reranking_documents(query, initial_docs, top_n=num_docs)
    
custom_retriever = CustomRetriever(vectorstore=vectordb)
llm = ChatOpenAI(temperature=0.2, model='gpt-4o')
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type='stuff',
    retriever = custom_retriever,
    return_source_documents = True
)

qa_chain.invoke("이 회사의 2022년 영업손실이 정확히 얼마야?")

{'query': '이 회사의 2022년 영업손실이 정확히 얼마야?',
 'result': '이 회사의 2022년 영업손실은 149.1억원입니다.',
 'source_documents': [Document(id='0885d6f9-3a33-4c13-b528-fa7dee2fe648', metadata={'producer': 'iText® 5.5.9 ©2000-2015 iText Group NV (AGPL-version)', 'creator': 'PyPDF', 'creationdate': '2024-06-26T16:15:14+09:00', 'moddate': '2024-06-26T16:15:14+09:00', 'source': './Data/투자설명서.pdf', 'total_pages': 514, 'page': 212, 'page_label': '213'}, page_content="이어, 당사는 2022년 GMP시설의 위탁생산계약 매출 약 4.8억원이 발생하였으나, 종업원급\n여 43.9억원, 유무형자산상각비 25.3억원이 발생하였으며, 연구개발 및 임상시험 지속 과정\n에서 지급수수료가 54.8억원 발생하는 등 총 영업비용 154억원이 발생하였습니다. 이에\n2022년 영업손실 149억원이 발생하였으며, 한편, 2022년 5월 금융감독원에서 공표한 '전환\n사채 콜옵션 회계처리' 감독지침에 따라 당사는 2021년 3월 19일 발행한 제2회 전환사채의"),
  Document(id='237a5d66-e1cb-4f91-a69f-2632cc3acfb1', metadata={'producer': 'iText® 5.5.9 ©2000-2015 iText Group NV (AGPL-version)', 'creator': 'PyPDF', 'creationdate': '2024-06-26T16:15:14+09:00', 'moddate': '2024-06-26T16:15:14+09:00', 'source': './Data/투자설명서.pdf', 'total_pages': 514, 

In [4]:
# 크로스 인코더 기반 리랭커
import os
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from pydantic import BaseModel, Field, ConfigDict
from langchain_community.docstore.document import Document
from typing import List, Dict, Any, Tuple
from sentence_transformers import CrossEncoder
from langchain_core.retrievers import BaseRetriever
from langchain_classic.chains import RetrievalQA

load_dotenv("./.env")
os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY")

file_path = "./Data/투자설명서.pdf"
loader = PyPDFLoader(file_path)
doc_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=100)
docs = loader.load_and_split(doc_splitter)

embedding = OpenAIEmbeddings(model='text-embedding-3-large')

faiss_store = FAISS.from_documents(docs, embedding)
persist_directory = "./faissDB/"
faiss_store.save_local(persist_directory)
vectordb = FAISS.load_local(persist_directory, embeddings=embedding, allow_dangerous_deserialization=True)

cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-12-v2')

c:\Users\Dohy\Desktop\dohy\RAG_master\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Dohy\.cache\huggingface\hub\models--cross-encoder--ms-marco-MiniLM-L-12-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 16422.94it/s]


In [8]:
# 벡터 기반 초기 검색과 크로스 인코더를 이용한 재순위화를 결합하여 더 정확한 문서 검색을 수행
class Retriever_with_cross_encoder(BaseRetriever):
    vectorstore: Any = Field(description="초기 검색을 위한 벡터 저장소")
    crossencoder: Any = Field(description="재순위화를 위한 크로스 인코더 모델")
    k: int = Field(default=5, description="초기에 검색할 문서 수")
    rerank_top_k: int = Field(default=2, description="재순위화 후 최종적으로 반환할 문서 수")

    model_config = ConfigDict(arbitrary_types_allowed=True)

    def _get_relevant_documents(self, query: str)->List[Document]: # 사용자의 질문을 입력받아 관련성 높은 문서 리스트를 반환
        # 초기 검색, 쿼리와 유사한 k개의 문서를 빠르게 검색. 관련 문서의 후보군을 추려내는 역할
        initial_docs = self.vectorstore.similarity_search(query, k=self.k)

        # 크로스 인코더용 쌍 준비
        pairs = [[query, doc.page_content] for doc in initial_docs]

        # 질문-문서 쌍의 관련성 점수 측정
        scores = self.crossencoder.predict(pairs)

        # 점수별 문서 정렬
        scored_docs = sorted(zip(initial_docs, scores), key=lambda x: x[1], reverse=True)

        # 상위 재순위화 문서 반환
        return [doc for doc, _ in scored_docs[:self.rerank_top_k]]

# 크로스 인코더 기반 검색기 생성
cross_encoder_retriever = Retriever_with_cross_encoder(
    vectorstore=vectordb, # 초기 밀집 검색에 사용
    crossencoder=cross_encoder,
    k=4,
    rerank_top_k=2
)

llm = ChatOpenAI(model='gpt-4o', temperature=0.2)

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=cross_encoder_retriever,
    return_source_documents=True
)

query = "이 회사의 2022년 영업손실이 정확히 얼마야?"
result = qa_chain({"query":query})

print(f"\n질문: {query}")
print(f"답변:{result['result']}")
print("\n답변 근거 문서:")
for i,doc in enumerate(result['source_documents']):
    print(f"\nDocument{i+1}")
    print(doc.page_content)


질문: 이 회사의 2022년 영업손실이 정확히 얼마야?
답변:이 회사의 2022년 영업손실은 149.1억원입니다.

답변 근거 문서:

Document1
이어, 당사는 2022년 GMP시설의 위탁생산계약 매출 약 4.8억원이 발생하였으나, 종업원급
여 43.9억원, 유무형자산상각비 25.3억원이 발생하였으며, 연구개발 및 임상시험 지속 과정
에서 지급수수료가 54.8억원 발생하는 등 총 영업비용 154억원이 발생하였습니다. 이에
2022년 영업손실 149억원이 발생하였으며, 한편, 2022년 5월 금융감독원에서 공표한 '전환
사채 콜옵션 회계처리' 감독지침에 따라 당사는 2021년 3월 19일 발행한 제2회 전환사채의

Document2
하여 2021년 영업손실 130.1억원, 2022년 영업손실 149.1억원, 2023년 영업손실 122억원, 2024년
1분기 영업손실 24.2억원이 발생하였습니다. 또한 영업 외적 측면에서도, 금융비용 등의 발생 영향
으로 인해 2021년 당기순손실 130.7억원, 2022년 당기순손실 228.7억원, 2023년 당기순손실
116.1억원, 2024년 1분기 당기순손실 32.9억원이 발생하는 등 지속적인 적자 구조를 면하지 못하고
있습니다.따라서 당사의 파이프라인에서 임상 성공을 통한 기술이전, 상품화 성공 등의 성과를 이루


In [17]:
# Self-RAG, 원래라면 오픈소스 언어 모델을 직접 학습시켜야하므로, 여기선 시뮬레이션만
import os
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from pydantic import BaseModel, Field
from langchain_classic.prompts import PromptTemplate
from typing import Literal

load_dotenv("./.env")
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
file_path = "./Data/투자설명서.pdf"
loader = PyPDFLoader(file_path)

doc_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=100)
docs = loader.load_and_split(doc_splitter)
embedding = OpenAIEmbeddings(model='text-embedding-3-large')
faiss_store = FAISS.from_documents(docs, embedding)
persist_directory = "./faissDB/"
faiss_store.save_local(persist_directory)

vectordb = FAISS.load_local(persist_directory, embeddings=embedding, allow_dangerous_deserialization=True)

# 검색 필요 여부 판단을 위한 컴포넌트
# 1. 출력 형식 클래스
# 2. 프롬프트 템플릿
# 3. 언어 모델

# 출력 형식 클래스, 언어 모델의 출력 형식을 명시하는 역할, 언어 모델이 해당 클래스의 내용에 맞게 출력하도록 강제
class RetrievalResponse(BaseModel):
    Reasoning: str = Field(description="검색의 필요 여부를 추론하는 과정(2~3 문장 이내)")
    Retrieve: Literal['Yes','No'] = Field(description='검색 필요 여부')

# 프롬프트 템플릿
retrieval_prompt = PromptTemplate(
    input_variables=["query"],
    template='''
    주어진 질문에 대해, 외부 문서를 참고하는 것이 더 나은 응답을 생성하는 데 도움이 되는지 판단해주세요. 추론 과정을
    작성한 뒤, "Yes" 또는 "No"로 답하세요

    다음 기준을 참고하세요:
    1. 사실적 정보나 복잡한 주제에 대한 상세한 설명을 요구하는 질문의 경우, 검색이 도움이 될 수 있습니다.
    2. 개인적인 의견, 창의적인 과제, 또는 간단한 계산의 경우, 일반적으로 검색이 필요하지 않습니다.
    3. 잘 알려진 사실에 대해서도, 검색은 때때로 추가적인 맥락이나 검증을 제공할 수 있습니다.

    질문: {query}
    '''
)

llm = ChatOpenAI(model='gpt-4o', max_completion_tokens=2000, temperature=0.2)
retrieval_chain = retrieval_prompt | llm.with_structured_output(RetrievalResponse)


# 관련성 평가 추론, 문서 검색을 수행한 후, 해당 문서와 질문의 연관성을 언어 모델을 활용해 다시 한번 평가
# 1. 출력 형식 클래스
# 2. 프롬프트 템플릿
# 3. 언어 모델

class RelevanceResponse(BaseModel):
    Reasoning: str = Field(description="연관 문서의 관련성 평가 추론 과정(2~3문장 이내)")
    ISREL: Literal['Relevant', 'Irrelevant'] = Field(description='관련성 평가 결과')

relevance_prompt = PromptTemplate(
    input_variables=['query','context'],
    template="""
    당신은 제공된 연관문서가 주어진 질문과 관련이 있는지, 그리고 질문에 답하는 데 유용한 정보를 제공하는지 판단하는 것입니다.
    만약 연관문서가 이 요구사항을 충족한다면 "Relevant"로 응답하고, 그렇지 않다면 "Irrelevant"로 응답하세요.

    다음 예시들을 참고하세요:

    예시 1:
    질문: 지구의 자전은 무엇을 야기하나요?
    연관문서: 자전은 낮과 밤의 순환을 야기하며, 이는 또한 온도와 습도의 상응하는 순환을 만듭니다. 지구가 자전함에 따라 해수면은 하루에 두 번 상승하고 하강합니다.
    Reasoning: 이 관련문서는 지구의 자전이 낮과 밤의 순환을 야기한다고 명시적으로 언급하고 있어, 질문에 직접적으로 관련이 있습니다.
    ISREL: Relevant

    예시 2:
    질문: 미국 하원의원 출마를 위한 나이 제한은 어떻게 되나요?
    연관문서: 헌법은 미국 상원 의원직을 위한 세 가지 자격 요건을 설정합니다: 나이(최소 30세), 미국 시민권(최소 9년), 그리고 선거 시점에 해당 상원의원이 대표하는 주의 거주자여야 합니다.
    Reasoning: 이 관련문서는 미국 하원이 아닌 상원 의원직에 대한 나이 제한을 논의하고 있어, 주어진 질문과 직접적인 관련이 없습니다.
    ISREL: Irrelevant

    위의 예시들을 참고하여, 다음 질문과 연관문서에 대해 평가해주세요.

    질문: {query}
    연관문서: {context}
"""
)

llm = ChatOpenAI(model='gpt-4o', max_completion_tokens=2000, temperature=0.2)
relevance_chain = relevance_prompt | llm.with_structured_output(RelevanceResponse)

# 검색 문서를 기반으로 답변 생성
# 1. 출력 형식 클래스
# 2. 프롬프트 템플릿
# 3. 언어 모델

class GenerationResponse(BaseModel):
    response: str = Field(description='질문과 연관 문서를 바탕으로 생성된 답변')

# 답변 생성 단계 프롬프트 템플릿
generation_prompt = PromptTemplate(
    input_variables=["query","context"],
    template="질문 '{query}'와 연관 문서 '{context}'를 기반으로 답변을 만들어주세요."
)
llm = ChatOpenAI(model='gpt-4o', max_completion_tokens=2000, temperature=0.2)
generation_chain = generation_prompt | llm.with_structured_output(GenerationResponse)

# 지원 평가
# 1. 출력 형식 클래스
# 2. 프롬프트 템플릿
# 3. 언어 모델
class SupportResponse(BaseModel):
    Reasoning: str = Field(description='답변이 연관 문서에 충분히 근거하는지 여부를 추론하는 과정(2~3문장 이내)')
    ISSUP: Literal['Fully supported', 'Partially supported', 'No supported'] = Field(description='답변이 연관 문서에 충분히 근거하는지에 대한 평가 결과')

support_prompt = PromptTemplate(
    input_variables=["query", "response", "context"],
    template="""
    당신은 주어진 답변이 연관문서의 정보에 얼마나 근거하고 있는지 평가하는 것입니다. 다음 척도를 사용하여 평가해주세요:

    1. Fully supported - 답변의 모든 정보가 연관문서에 의해 뒷받침되거나, 연관문서에서 직접 추출된 경우입니다. 이는 답변과 연관문서의 일부가 거의 동일한 극단적인 경우에만 해당합니다.
    2. Partially supported - 답변이 어느 정도 연관문서에 의해 뒷받침되지만, 연관문서에서 다루지 않는 주요 정보가 답변에 포함된 경우입니다. 예를 들어, 질문이 두 가지 개념에 대해 물었는데 연관문서가 그 중 하나만 다루고 있다면 이에 해당합니다.
    3. No support - 답변이 연관문서를 완전히 무시하거나, 관련이 없거나, 또는 연관문서와 모순되는 경우입니다. 연관문서가 질문과 무관한 경우에도 이에 해당할 수 있습니다.

    주의: 답변이 사실인지 아닌지를 판단하기 위해 외부 정보나 지식을 사용하지 마세요. 오직 답변이 연관문서에 의해 뒷받침되는지만 확인하세요. 답변이 질문을 잘 따르고 있는지는 판단하지 않습니다.

    다음 예시를 참고하세요:
    질문: 자연어 처리에서 단어 임베딩의 사용에 대해 설명해주세요.
    답변: 단어 임베딩은 감성 분석, 텍스트 분류, 다음 단어 예측, 동의어와 유추 관계 이해 등의 작업에 유용합니다.
    연관문서: 단어 임베딩은 자연어 처리(NLP)에서 어휘의 단어나 구를 실수 벡터에 매핑하는 언어 모델링 및 특징 학습 기술의 총칭입니다. 단어와 구 임베딩은 기본 입력 표현으로 사용될 때 구문 분석, 감성 분석, 다음 토큰 예측, 유추 감지 등의 NLP 작업에서 성능 향상을 보여주었습니다.
    Reasoning: 답변에서 언급된 단어 임베딩의 모든 응용 분야(감성 분석, 텍스트 분류, 다음 단어 예측, 동의어와 유추 관계 이해)가 연관문서에서 직접적으로 언급되거나 유추될 수 있습니다. 따라서 답변은 연관문서에 의해 완전히 뒷받침됩니다.
    ISSUP: Fully supported

    위의 예시를 참고하여, 주어진 질문, 답변, 연관문서에 대한 당신의 평가를 제시해주세요:

    질문: {query}
    답변: {response}
    연관문서: {context}
"""
)
llm = ChatOpenAI(model="gpt-4o", max_tokens=2000, temperature=0.2)
support_chain = relevance_prompt | llm.with_structured_output(SupportResponse)

# 유용성 평가
# 1. 출력 형식 클래스
# 2. 프롬프트 템플릿
# 3. 언어 모델
class UtilityResponse(BaseModel):
    Reasoning: str = Field(description="응답의 유용성 평가 추론과정")
    ISUSE: Literal[1, 2, 3, 4, 5] = Field(description="응답의 유용성 평가결과")

utility_prompt = PromptTemplate(
    input_variables=["query", "response"],
    template="""
    주어진 질문과 답변에 대해, 그 응답이 얼마나 도움이 되고 유익한 답변인지 1점(최저)부터 5점(최고)까지 평가해주세요. 이 점수를 'Utility_score'라고 부릅니다.

    평가 기준은 다음과 같습니다:
    5: 답변이 완벽하고 매우 상세하며 정보가 풍부하여 질문의 정보 요구를 완전히 충족시킵니다.
    4: 답변이 대체로 질문의 요구를 충족시키지만, 더 자세한 정보 제공, 응답 구조 개선, 또는 일관성 향상 등의 약간의 개선이 가능합니다.
    3: 답변이 수용 가능하지만, 사용자의 요구를 만족시키기 위해 주요한 추가 정보나 개선이 필요합니다.
    2: 답변이 주요 요청을 다루고는 있지만, 불완전하거나 질문과 완전히 관련이 없습니다.
    1: 답변이 거의 주제와 관련이 없거나 완전히 무관합니다.

    다음 예시들을 참고하세요:

    예시 1:
    질문: 2023년 현재 영국의 총리는 누구인가요?
    답변: Boris Johnson은 2019년부터 2022년까지 영국의 총리였습니다.
    Reasoning: 이 응답은 2019년부터 2022년까지의 영국 총리에 대해 사실적으로 정확한 진술을 제공하지만, 질문은 2023년 현재의 총리를 묻고 있습니다. 따라서 질문에 직접적으로 답하지 않아 유용성이 2점입니다.
    ISUSE: 2

    예시 2:
    질문: 여행 목적지인 도쿄, 일본에 대한 설명을 바탕으로 10개의 관광 명소를 추천하고 각각에 대해 자세히 설명해주세요.
    답변: 도쿄는 흥미진진한 관광 명소로 가득한 활기찬 도시입니다. 꼭 봐야 할 명소로는 도쿄 스카이트리, 도쿄 디즈니랜드, 센소지 사원, 메이지 신궁, 츠키지 어시장, 하라주쿠, 신주쿠 교엔 등이 있습니다.
    Reasoning: 이 응답은 각 명소에 대한 설명을 제공하지 않았고, 명소의 수도 10개보다 적습니다. 질문에 부분적으로 답변하고 있지만, 지시사항을 엄격히 따르지 않았습니다.
    ISUSE: 3

    위의 예시들을 참고하여, 주어진 질문과 응답에 대한 당신의 평가를 제시해주세요:

    질문: {query}
    응답: {response}
"""
)
llm = ChatOpenAI(model="gpt-4o", max_tokens=2000, temperature=0.2)
utility_chain = utility_prompt | llm.with_structured_output(UtilityResponse)

In [19]:
class SelfRAG:
    def __init__(self, vectorstore, retrieval_chain, relevance_chain, generation_chain, support_chain, utility_chain, top_k):
        self.vectorstore = vectorstore
        self.retrieval_chain = retrieval_chain
        self.relevance_chain = relevance_chain
        self.generation_chain = generation_chain
        self.support_chain = support_chain
        self.utility_chain = utility_chain
        self.top_k = top_k

    # 검색 필요 여부 판단 과정
    def determine_retrieval(self, query):
        print("\n1단계: 검색 필요 여부 결정 중...")
        input_data={"query":query}
        retrieval_decision_response = self.retrieval_chain.invoke(input_data)
        reasoning = retrieval_decision_response.Reasoning
        retrieve_token = retrieval_decision_response.Retrieve
        print(f"검색 결정 추론 과정: {reasoning}")
        print(f"검색 결정: {retrieve_token}")
        return retrieve_token
    # 관련 문서 검색
    def retrieve_documents(self, query):
        print("\n2단계: 관련 문서 검색 중...")
        docs = self.vectorstore.similarity_search(query, k=self.top_k)
        contexts = [doc.page_content for doc in docs]
        print(f"{len(contexts)}개의 문서를 검색했습니다")
        return contexts
    # 문서의 관련성 평가
    def evaluate_relevance(self, query, contexts):
        print("\n3단계: 문서의 관련성 평가 중...")
        relevant_contexts=[]
        for i, context in enumerate(contexts):
            input_data = {'query':query, 'context':context}
            relevance_response = self.relevance_chain.invoke(input_data)
            relevance_reasoning = relevance_response.Reasoning
            relevance_token = relevance_response.ISREL
            print(f"문서 {i+1} 관련성 추론 과정: {relevance_reasoning}")
            print(f"문서 {i+1} 관련성: {relevance_token}")
            if relevance_token == "Relevant":
                relevant_contexts.append(context)

        print(f"관련된 컨텍스트 수: {len(relevant_contexts)}")
        return relevant_contexts
    # 관련 컨텍스트로 응답 생성
    def generate_responses(self, query, relevant_contexts):
        print(f"\n4단계: 관련 컨텍스트로 응답 생성 중...")
        responses = []
        for i, context in enumerate(relevant_contexts):
            print(f"컨텍스트 {i+1}에 대한 응답 생성 중...")
            input_data = {'query':query, 'context':context}
            response = self.generation_chain.invoke(input_data).response
            responses.append(response)
        return responses
    # 관련된 컨텍스트가 없거나, 검색이 필요하지 않다고 판단된 경우에 사용
    def generate_without_retrieval(self, query):
        input_data = {'query':query, 'context':'관련된 컨텍스트를 찾지 못했습니다.'}
        response = self.generation_chain.invoke(input_data).response
        return response
    # 생성된 응답에 대해 지원 평가와 유용성 평가 수행
    def assess_and_evalute(self, query, responses, relevnat_contexts):
        assessed_responses= []
        for i, (response, context) in enumerate(zip(responses, relevnat_contexts)):
            # 지원 평가
            print(f"\n5단계: 응답 {i+1}의 지원 평가 중...")
            input_data={'query':query, 'response':response, 'context':context}
            support_response = self.support_chain.invoke(input_data)
            support_reasoning = support_response.Reasoning
            support_token = support_response.ISSUP
            print(f"지원 평가 추론 과정: {support_reasoning}")
            print(f"지원 평가: {support_token}")

            # 유용성 평가
            print(f"\n6단계: 응답 {i+1}의 유용성 평가 중...")
            input_data = {'query':query, 'response':response}
            utility_response = self.utility_chain.invoke(input_data)
            utility_reasoning = utility_response.Reasoning
            utility_token = int(utility_response.ISUSE)
            print(f"유용성 점수 평가과정: {utility_reasoning}")
            print(f"유용성 점수: {utility_token}")
            assessed_responses.append((response, support_token, utility_token))

        return assessed_responses
    # 평가된 응답들 중 가장 우수한 응답 선택
    def select_best_response(self, responses):
        print("\n최고의 응답 선택 중..")
        # 1. Fully supported가 있는지 확인
        fully_supported = [r for r in responses if r[1] == 'Fully suppported']
        if fully_supported:
            best_response = max(fully_supported, key=lambda x: x[2])
            print(f"선택된 응답의 지원 상태: {best_response[1]}, 유용성 점수: {best_response[2]}")
            return best_response
        
        # 2. Fully_supported가 없다면 Partially supported 확인
        partially_supported = [f for r in responses if r[1] == 'Partially supported']
        if partially_supported:
            best_response = max(partially_supported, key=lambda x: x[2])
            print(f"선택된 응답의 지원 상태: {best_response[1]}, 유용성 점수: {best_response[2]}")
            return best_response
        
        # 3. 둘 다 없는 경우, 유용성 점수(x[2]) 기준으로 선택
        best_response = max(responses, key=lambda x:x[2])
        print(f"선택된 응답의 지원 상태: {best_response[1]}, 유용성 점수: {best_response[2]}")
        return best_response
    # Self-RAG 시스템의 전체 워크플로우를 통합하고 조율
    def process_query(self, query):
        print(f"\n쿼리 처리 중: {query}")

        # 1단계: 검색이 필요한지 결정
        retrieval_decision = self.determine_retrieval(query)
        if retrieval_decision == 'Yes':
            # 2단계: 관련 문서 검색
            contexts = self.retrieve_documents(query)

            # 3단계: 검색된 문서의 관련성 평가
            relevant_contexts = self.evaluate_relevance(query, contexts)

            if not relevant_contexts:
                # 관련된 컨텍스트가 없으면 검색 없이 생성
                print("관련된 컨텍스트를 찾지 못했습니다. 검색 없이 생성합니다...")
                return self.generate_without_retrieval(query)
            
            # 4단계: 관련 컨텍스트를 사용하여 응답 생성
            responses = self.generate_responses(query, relevant_contexts)

            # 5,6단계: 지원 평가 및 유용성 평가
            assessed_responses = self.assess_and_evalute(query, responses, relevant_contexts)

            #최고의 응답 선택
            best_response = self.select_best_response(assessed_responses)
            return best_response[0]
        
        else:
            # 검색 없이 생성
            print("검색 없이 생성합니다...")
            return self.generate_without_retrieval(query)
    

In [20]:
self_rag_instance = SelfRAG(
    vectorstore=vectordb,
    retrieval_chain = retrieval_chain,
    relevance_chain = relevance_chain,
    generation_chain = generation_chain,
    support_chain = support_chain,
    utility_chain = utility_chain,
    top_k=4
)

query = "이 회사의 바이오 의약품 라이센스 아웃 수익을 알려줘"
response = self_rag_instance.process_query(query)

print('\n최종 응답:')
print(response)


쿼리 처리 중: 이 회사의 바이오 의약품 라이센스 아웃 수익을 알려줘

1단계: 검색 필요 여부 결정 중...
검색 결정 추론 과정: 질문은 특정 회사의 바이오 의약품 라이센스 아웃 수익에 대한 정보를 요구하고 있습니다. 이는 특정한 회사의 재무 정보에 해당하며, 일반적으로 공개된 데이터베이스나 회사의 공식 보고서에서 확인할 수 있는 정보입니다. 따라서, 외부 문서를 참고하여 정확한 수익 정보를 제공하는 것이 필요합니다.
검색 결정: Yes

2단계: 관련 문서 검색 중...
4개의 문서를 검색했습니다

3단계: 문서의 관련성 평가 중...
문서 1 관련성 추론 과정: The provided document discusses the termination of a licensing agreement related to a biopharmaceutical product. However, it does not provide any information about the revenue generated from the licensing out of biopharmaceuticals. The document focuses on the termination details and where to find more information about the contract, rather than the financial aspects of the licensing agreement.
문서 1 관련성: Irrelevant
문서 2 관련성 추론 과정: The provided document mentions a license agreement related to a personalized cancer immunotherapy vaccine. However, it does not provide specific information about the revenue generated from licensing out biopharmaceuticals. The document focuse